# Aula 01 · Figuras com dados reais

Gera os dados de duas figuras da Aula 01:

1. **Perceptron no MNIST (0 vs 1)**: a regra de Rosenblatt, pesos começando em zero, 10 épocas. Salva os erros por época e os 784 pesos.
2. **MLP 784 → 16 → 10**: treinado por 8 épocas com Adam. Salva os pesos da camada oculta, que viram as imagens do slide "O que uma rede aprende sozinha".

As figuras SVG são montadas por `tools/aula01_figuras_svg.py`.

In [ ]:
import json, os, numpy as np, torch
from torchvision import datasets
from matplotlib import pyplot as plt, colors

OUT, PNG = 'dados', 'png'
os.makedirs(OUT, exist_ok=True); os.makedirs(PNG, exist_ok=True)
tr = datasets.MNIST('.', train=True, download=True)
te = datasets.MNIST('.', train=False, download=True)
Xtr, ytr = tr.data.reshape(-1, 784).numpy() / 255., tr.targets.numpy()
Xte, yte = te.data.reshape(-1, 784).numpy() / 255., te.targets.numpy()

In [ ]:
# 1. Perceptron 0 vs 1: w <- w + y x quando erra (y em {-1, +1}); o viés é uma entrada constante 1
rng = np.random.default_rng(0)
m = ytr < 2; X = np.c_[Xtr[m], np.ones(m.sum())]; y = np.where(ytr[m] == 1, 1, -1)
mt = yte < 2; Xt = np.c_[Xte[mt], np.ones(mt.sum())]; yt = np.where(yte[mt] == 1, 1, -1)
w = np.zeros(785)
err = lambda X, y: float(np.mean(np.sign(X @ w + 1e-12) != y))
hist = [(0, err(X, y), err(Xt, yt), 0)]
for ep in range(1, 11):
    upd = 0
    for i in rng.permutation(len(y)):
        if y[i] * (X[i] @ w) <= 0:
            w += y[i] * X[i]; upd += 1
    hist.append((ep, err(X, y), err(Xt, yt), upd))
    print(f'época {ep}: {upd} correções, erro treino {hist[-1][1]:.4%}, erro teste {hist[-1][2]:.4%}')
np.save(f'{OUT}/perc_w.npy', w[:784])

In [ ]:
# 2. MLP 784 -> 16 -> 10
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(784, 16), torch.nn.ReLU(), torch.nn.Linear(16, 10))
opt = torch.optim.Adam(net.parameters(), 1e-3)
Xa = torch.tensor(Xtr, dtype=torch.float32); ya = torch.tensor(ytr)
for ep in range(8):
    p = torch.randperm(len(ya))
    for k in range(0, len(ya), 128):
        b = p[k:k + 128]
        loss = torch.nn.functional.cross_entropy(net(Xa[b]), ya[b])
        opt.zero_grad(); loss.backward(); opt.step()
acc = (net(torch.tensor(Xte, dtype=torch.float32)).argmax(1).numpy() == yte).mean()
print('acurácia no teste:', acc)
np.save(f'{OUT}/mlp_w1.npy', net[0].weight.detach().numpy())
json.dump({'perc_mnist': hist, 'mlp_acc': float(acc)}, open(f'{OUT}/data.json', 'w'))

In [ ]:
# Imagens: pesos (azul = positivo, vermelho = negativo) e alguns dígitos
cm = colors.LinearSegmentedColormap.from_list('rb', ['#e53e3e', '#ffffff', '#004b87'])
def save(a, name, cmap, vmin=None, vmax=None, scale=8):
    plt.imsave(f'{PNG}/{name}.png', np.kron(a, np.ones((scale, scale))), cmap=cmap, vmin=vmin, vmax=vmax)
wp = np.load(f'{OUT}/perc_w.npy').reshape(28, 28); v = np.abs(wp).max(); save(wp, 'perc_weights', cm, -v, v)
for i, r in enumerate(np.load(f'{OUT}/mlp_w1.npy')):
    r = r.reshape(28, 28); v = np.abs(r).max(); save(r, f'mlp_w{i:02d}', cm, -v, v, scale=4)
for d in range(10):
    for j, idx in enumerate(np.where(ytr == d)[0][:6]):
        save(Xtr[idx].reshape(28, 28), f'digit_{d}_{j}', 'gray_r', 0, 1, scale=4)
print('pronto: rode  python tools/aula01_figuras_svg.py dados png slides/img/aula01')